# Study 25 - Do co-located pollutants carry onset information, without changing any parameter?

**Why.** study24 showed that on the Beijing benchmark the estimator with the Methods configuration (PM2.5, wind,
temperature) is below forward fill (0.667 vs 0.720), and that a variant with all six pollutants is above it (0.728).
That variant used weights 1.4 / 0.8 taken from the feasibility study, so it changes parameters. The question for the
paper (Edison, 18 Nov) is whether the improvement holds when **only the set of inputs changes**.

**Settings fixed before running (not tuned on the results):**
- Extra pollutants follow the rule already stated in the Methods: "pollutants keep their sign", so orientation +1 and
  weight 1.0, the same as PM2.5. No exception for ozone; check 2 shows whether ozone helps or hurts.
- Wind (-1, 0.5) and temperature (-1, 0.4) as in the Methods. Constants, transitions and prior unchanged.
- Nordic NO2 is treated exactly like PM2.5: log(1+c), normalised per station from the training (even) months,
  orientation +1, weight 1.0. The implausible-value filter is the one used for PM2.5 (0 to 1500).
- Test months (odd) for all metrics; paired bootstrap, 2000 resamples, on the difference.

**Checks**
1. Logistic model with and without the extra pollutants (Beijing) and with and without NO2 (Nordic): is the
   information in the data, independent of the estimator?
2. Beijing: add one pollutant at a time to the Methods configuration, and remove one at a time from the six-pollutant
   configuration: which pollutants carry the signal?
3. Beijing: six pollutants with the Methods weights, complete data, under PM2.5 loss, and calibration.
4. Nordic: the specified configuration plus NO2, complete data and under PM2.5 loss.

Built-in checks: the Methods configuration must reproduce 0.667 (Beijing, study24) and 0.794 (Nordic, Table 1);
the study24 exploratory variant must reproduce 0.728.

cwd = `notebooks/`. Outputs -> `../outputs/study25_pollutants/`.

In [1]:
import glob, math, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
np.random.seed(20261118)
OUT=Path("../outputs/study25_pollutants"); OUT.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({"figure.dpi":130,"savefig.dpi":200,"font.size":11,"axes.grid":True,"grid.alpha":0.25,
                     "axes.spines.top":False,"axes.spines.right":False})
# ---- estimator: the study24 filter for a list of (signal, orientation, weight); it reproduced the study21 estimator ----
K=5; LEV=np.linspace(0,1,K); SDK=1.0+0.5*LEV; LOGC=0.5*np.log(2*np.pi)
def lgv(z): return -LOGC-np.log(SDK)-0.5*((z-2.0*LEV)/SDK)**2
A_LO,A_HI=np.zeros((K,K)),np.zeros((K,K))
for A_,pdn in [(A_LO,0.05),(A_HI,0.15)]:
    for k in range(K):
        up=0.25 if k<K-1 else 0; dn=pdn if k>0 else 0; A_[k,k]=1-up-dn
        if k<K-1: A_[k,k+1]=up
        if k>0: A_[k,k-1]=dn
def twin_cfg(Z,cfg):
    n=len(next(iter(Z.values()))); prior=np.exp(-3*np.arange(K)); logb=np.log(prior/prior.sum()+1e-12); risk=np.empty(n)
    cols=[(Z[v],o,w) for v,o,w in cfg]
    for t in range(n):
        b=np.exp(logb-logb.max()); b/=b.sum(); el=(b*LEV).sum()
        logbp=np.log(b@(A_LO if el>0.4 else A_HI)+1e-12); ll=np.zeros(K)
        for zv,o,w in cols:
            z=zv[t]
            if np.isnan(z): continue
            ll+=w*lgv(z*o)
        logt=logbp+ll; logb=logt-(logt.max()+np.log(np.exp(logt-logt.max()).sum())); risk[t]=float((np.exp(logb)*LEV).sum())
    return risk
# ---- metrics and helpers (as study24) ----
def auc(s,yv):
    s=np.asarray(s,float);yv=np.asarray(yv,int);pos,neg=s[yv==1],s[yv==0]
    if len(pos)==0 or len(neg)==0:return float('nan')
    o=np.argsort(s);r=np.empty(len(s));r[o]=np.arange(1,len(s)+1)
    _,inv,c=np.unique(s,return_inverse=True,return_counts=True);ss=np.zeros(len(c));np.add.at(ss,inv,r);r=(ss/c)[inv]
    return float((r[yv==1].sum()-len(pos)*(len(pos)+1)/2)/(len(pos)*len(neg)))
def apr(s,y):
    s=np.asarray(s,float);y=np.asarray(y,int);o=np.argsort(-s);y=y[o];tp=np.cumsum(y);fp=np.cumsum(1-y);prec=tp/(tp+fp);rec=tp/max(y.sum(),1)
    return float(np.sum((rec-np.r_[0,rec[:-1]])*prec))
def brier(p,y): return float(np.mean((np.asarray(p,float)-np.asarray(y,float))**2))
def ece(p,y,nb=10):
    p=np.asarray(p,float); y=np.asarray(y,float); ed=np.linspace(0,1,nb+1); e=0.0
    for i in range(nb):
        m=(p>=ed[i])&(p<ed[i+1] if i<nb-1 else p<=ed[i+1])
        if m.sum(): e+=m.mean()*abs(y[m].mean()-p[m].mean())
    return float(e)
def sig(z): return 1/(1+np.exp(-np.clip(z,-30,30)))
def platt_fit(x,y,ep=3000,lr=.3):
    x=np.asarray(x,float);mu,sd=x.mean(),x.std()+1e-9;xs=(x-mu)/sd;w=0.;bi=0.
    for _ in range(ep): p=1/(1+np.exp(-(w*xs+bi)));w-=lr*np.mean((p-y)*xs);bi-=lr*np.mean(p-y)
    return lambda z:1/(1+np.exp(-(w*((np.asarray(z,float)-mu)/sd)+bi)))
def fitlog(X,y,ep=1500,lr=.25,l2=1.):
    X=np.asarray(X,float);mu=X.mean(0);sd=X.std(0)+1e-9;Xb=np.hstack([np.ones((len(X),1)),(X-mu)/sd]);w=np.zeros(Xb.shape[1])
    for _ in range(ep): p=sig(Xb@w);g=Xb.T@(p-y)/len(y);g[1:]+=l2/len(y)*w[1:];w-=lr*g
    return (w,mu,sd)
def predlog(m,X): w,mu,sd=m; return sig(np.hstack([np.ones((len(X),1)),(np.asarray(X,float)-mu)/sd])@w)
def ffill(a):
    out=a.copy(); last=np.nan
    for i in range(len(out)):
        if np.isnan(out[i]): out[i]=last
        else: last=out[i]
    return out
def paired_boot(sa,sb,y,n=2000,seed=1):
    sa=np.asarray(sa,float); sb=np.asarray(sb,float); y=np.asarray(y,int); rng=np.random.default_rng(seed); d=[]
    for _ in range(n):
        i=rng.integers(0,len(y),len(y)); yi=y[i]
        if 0<yi.sum()<len(yi): d.append(auc(sa[i],yi)-auc(sb[i],yi))
    lo,hi=np.percentile(d,[2.5,97.5]); return float(auc(sa,y)-auc(sb,y)),float(lo),float(hi)
def fmt(d,lo,hi): return f"{d:+.3f} [{lo:+.3f}, {hi:+.3f}]"
RESULTS=[]   # one row per reported comparison, saved at the end
def keep(check,what,a,b,d=None,lo=None,hi=None):
    RESULTS.append(dict(check=check,comparison=what,value_a=a,value_b=b,diff=d,lo95=lo,hi95=hi))
print("helpers ready")

/Users/sumiyasayeed/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
/Users/sumiyasayeed/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


helpers ready


## Part A - Beijing (12 stations, threshold 75 ug/m3, six-hour means, 24 h horizon; as study24)

In [2]:
BASE="../data/beijing_air/PRSA_Data_20130301-20170228"
POLL=["PM2.5","PM10","SO2","NO2","CO","O3"]; MET=["TEMP","PRES","DEWP","WSPM"]; ALL=POLL+MET; EXTRA=POLL[1:]
TAU_B,STEP,H=75.0,6,4
def onset_index(pm,keep,tau):
    out=[]
    for t in range(2,len(pm)-H):
        if np.isnan(pm[t]) or pm[t]>=tau or not keep[t]: continue
        fut=pm[t+1:t+1+H]
        if np.all(np.isnan(fut)): continue
        out.append((t,int(np.nanmax(fut)>=tau)))
    return out
BJ={}
for f in sorted(glob.glob(BASE+"/PRSA_Data_*.csv")):
    d=pd.read_csv(f); d["dt"]=pd.to_datetime(d[["year","month","day","hour"]]); d=d.set_index("dt").sort_index()
    g=d[ALL].resample(f"{STEP}h").mean(); tm=(g.index.month%2==0)
    Z={}
    for c in ALL:
        v=g[c].astype(float).to_numpy()
        if c in POLL: v=np.log1p(np.clip(v,0,None))
        mu,sd=np.nanmean(v[tm]),np.nanstd(v[tm])+1e-9; Z[c]=(v-mu)/sd
    pm=g["PM2.5"].to_numpy()
    BJ[d["station"].iloc[0]]=dict(Z=Z,pm=pm,tm=tm,te=onset_index(pm,~tm,TAU_B),tr=onset_index(pm,tm,TAU_B))
print(len(BJ),"Beijing stations | test rows",sum(len(d["te"]) for d in BJ.values()),"| train rows",sum(len(d["tr"]) for d in BJ.values()))

METHODS=[("PM2.5",1,1.0),("WSPM",-1,0.5),("TEMP",-1,0.4)]
M6=METHODS+[(p,1,1.0) for p in EXTRA]                                      # check 3: Methods rule, no new parameter
S24=[(p,1,1.4) for p in POLL]+[("WSPM",-1,0.8),("TEMP",-1,0.8)]            # study24 exploratory variant (reference)
def score_bj(cfg,frac=0.0,seed=None,rows="te"):
    rng=np.random.default_rng(seed) if frac>0 else None; E=[];C=[];Y=[]
    for st,d in BJ.items():
        Z=dict(d["Z"]); zpm=Z["PM2.5"].copy()
        if frac>0: zpm[rng.random(len(zpm))<frac]=np.nan
        Z["PM2.5"]=zpm; risk=twin_cfg(Z,cfg); cur=ffill(zpm)
        for t,y in d[rows]:
            E.append(risk[t]); C.append(cur[t] if not np.isnan(cur[t]) else 0.0); Y.append(y)
    return np.array(E),np.array(C),np.array(Y,int)
E_meth,C0,Y0=score_bj(METHODS); E_s24,_,_=score_bj(S24)
print(f"check: Methods configuration {auc(E_meth,Y0):.3f} (study24: 0.667) | study24 variant {auc(E_s24,Y0):.3f} (study24: 0.728) | forward fill {auc(C0,Y0):.3f} (0.720)")

12 Beijing stations | test rows 20707 | train rows 21008
check: Methods configuration 0.667 (study24: 0.667) | study24 variant 0.728 (study24: 0.728) | forward fill 0.720 (0.720)


### Check 1 (Beijing) - logistic model with and without the five extra pollutants
Current values only (NaN set to 0), fitted on the training months and scored on the test months.

In [3]:
def bj_X(rows,cols):
    X=[];Y=[]
    for d in BJ.values():
        for t,y in d[rows]: X.append([np.nan_to_num(d["Z"][c][t]) for c in cols]); Y.append(y)
    return np.array(X,float),np.array(Y,int)
F3=["PM2.5","WSPM","TEMP"]; F8=F3+EXTRA
X3tr,ytr=bj_X("tr",F3); X3te,yte=bj_X("te",F3); X8tr,_=bj_X("tr",F8); X8te,_=bj_X("te",F8); X10tr,_=bj_X("tr",ALL); X10te,_=bj_X("te",ALL)
p3=predlog(fitlog(X3tr,ytr),X3te); p8=predlog(fitlog(X8tr,ytr),X8te); p10=predlog(fitlog(X10tr,ytr),X10te)
d,lo,hi=paired_boot(p8,p3,yte)
print(f"logistic, PM2.5 + wind + temperature : AUROC {auc(p3,yte):.3f}  AUPRC {apr(p3,yte):.3f}")
print(f"logistic, + five extra pollutants    : AUROC {auc(p8,yte):.3f}  AUPRC {apr(p8,yte):.3f}   difference {fmt(d,lo,hi)}")
print(f"logistic, all ten variables (ref.)   : AUROC {auc(p10,yte):.3f}")
keep("1 Beijing logistic","+5 pollutants vs 3 signals",auc(p8,yte),auc(p3,yte),d,lo,hi)

logistic, PM2.5 + wind + temperature : AUROC 0.736  AUPRC 0.645
logistic, + five extra pollutants    : AUROC 0.754  AUPRC 0.662   difference +0.018 [+0.015, +0.021]
logistic, all ten variables (ref.)   : AUROC 0.762


### Check 2 (Beijing) - which pollutants carry the signal
Add one pollutant to the Methods configuration (+1, weight 1.0), and remove one from the six-pollutant configuration.
Intervals are given for the add-one comparisons.

In [4]:
E_m6,_,_=score_bj(M6)
rows=[]
for p in EXTRA:
    E_add,_,_=score_bj(METHODS+[(p,1,1.0)]); d,lo,hi=paired_boot(E_add,E_meth,Y0)
    E_drop,_,_=score_bj([x for x in M6 if x[0]!=p])
    rows.append(dict(pollutant=p,add_AUROC=auc(E_add,Y0),add_gain=d,add_lo95=lo,add_hi95=hi,
                     drop_AUROC=auc(E_drop,Y0),drop_change=auc(E_drop,Y0)-auc(E_m6,Y0)))
    keep("2 Beijing add-one",f"Methods + {p} vs Methods",auc(E_add,Y0),auc(E_meth,Y0),d,lo,hi)
AD=pd.DataFrame(rows); AD.to_csv(OUT/"s25_beijing_add_drop.csv",index=False)
print(f"Methods configuration {auc(E_meth,Y0):.3f} | six pollutants (Methods weights) {auc(E_m6,Y0):.3f}\n")
for r in rows:
    print(f"{r['pollutant']:5s} | add to Methods: {r['add_AUROC']:.3f} gain {fmt(r['add_gain'],r['add_lo95'],r['add_hi95'])}"
          f" | remove from six: {r['drop_AUROC']:.3f} change {r['drop_change']:+.3f}")

Methods configuration 0.667 | six pollutants (Methods weights) 0.724

PM10  | add to Methods: 0.684 gain +0.017 [+0.013, +0.021] | remove from six: 0.728 change +0.004
SO2   | add to Methods: 0.695 gain +0.028 [+0.024, +0.033] | remove from six: 0.709 change -0.015
NO2   | add to Methods: 0.684 gain +0.018 [+0.014, +0.022] | remove from six: 0.714 change -0.010
CO    | add to Methods: 0.678 gain +0.011 [+0.008, +0.014] | remove from six: 0.720 change -0.004
O3    | add to Methods: 0.644 gain -0.023 [-0.030, -0.015] | remove from six: 0.711 change -0.013


### Check 3 (Beijing) - six pollutants with the Methods weights
Complete data against forward fill and against the Methods configuration; then PM2.5 masked independently with the
same levels and seeds as study24 (one realisation at 0 %, three otherwise), so the masks equal those of study24.
The Methods-configuration column is read from study24.

In [5]:
d,lo,hi=paired_boot(E_m6,C0,Y0); keep("3 Beijing six pollutants","six (Methods weights) vs forward fill, complete",auc(E_m6,Y0),auc(C0,Y0),d,lo,hi)
d2,lo2,hi2=paired_boot(E_m6,E_meth,Y0); keep("3 Beijing six pollutants","six (Methods weights) vs Methods, complete",auc(E_m6,Y0),auc(E_meth,Y0),d2,lo2,hi2)
d3,lo3,hi3=paired_boot(E_m6,E_s24,Y0); keep("3 Beijing six pollutants","six (Methods weights) vs study24 variant, complete",auc(E_m6,Y0),auc(E_s24,Y0),d3,lo3,hi3)
print(f"complete data: six pollutants {auc(E_m6,Y0):.3f} (AUPRC {apr(E_m6,Y0):.3f}) | forward fill {auc(C0,Y0):.3f} (AUPRC {apr(C0,Y0):.3f})")
print(f"  vs forward fill   {fmt(d,lo,hi)}\n  vs Methods config {fmt(d2,lo2,hi2)}\n  vs study24 variant {fmt(d3,lo3,hi3)}\n")
S24CSV=Path("../outputs/study24_beijing_aligned/s24_beijing_dropout.csv")
REF=pd.read_csv(S24CSV).set_index("dropout") if S24CSV.exists() else None
FR=[0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.85,0.9]; res=[]
for fr in FR:
    seeds=[30] if fr==0 else [30,31,32]; Mv=[]; dd=lo_=hi_=np.nan
    for s in seeds:
        E,C,Y=score_bj(M6,fr,s); Mv.append((auc(E,Y),apr(E,Y),auc(C,Y),apr(C,Y)))
        if s==30 and fr in (0.5,0.85): dd,lo_,hi_=paired_boot(E,C,Y)
    m=np.mean(Mv,axis=0); meth=REF.loc[fr,"est_AUROC"] if REF is not None else np.nan
    res.append(dict(dropout=fr,six_AUROC=m[0],six_AUPRC=m[1],LOCF_AUROC=m[2],LOCF_AUPRC=m[3],adv=m[0]-m[2],
                    adv_seed30=dd,adv_lo95=lo_,adv_hi95=hi_,methods_AUROC=meth))
    ci="" if np.isnan(lo_) else f" [seed30 {lo_:+.3f}, {hi_:+.3f}]"
    print(f"{fr*100:3.0f}% | six pollutants {m[0]:.3f} (AUPRC {m[1]:.3f}) | forward fill {m[2]:.3f} | adv {m[0]-m[2]:+.3f}{ci} | Methods config {meth:.3f}")
SIX=pd.DataFrame(res); SIX.to_csv(OUT/"s25_beijing_six_dropout.csv",index=False)

complete data: six pollutants 0.724 (AUPRC 0.598) | forward fill 0.720 (AUPRC 0.618)
  vs forward fill   +0.005 [-0.001, +0.010]
  vs Methods config +0.058 [+0.051, +0.064]
  vs study24 variant -0.004 [-0.005, -0.003]

  0% | six pollutants 0.724 (AUPRC 0.598) | forward fill 0.720 | adv +0.005 | Methods config 0.667
 10% | six pollutants 0.722 (AUPRC 0.596) | forward fill 0.708 | adv +0.014 | Methods config 0.652
 20% | six pollutants 0.720 (AUPRC 0.595) | forward fill 0.696 | adv +0.025 | Methods config 0.642
 30% | six pollutants 0.718 (AUPRC 0.593) | forward fill 0.681 | adv +0.037 | Methods config 0.632
 40% | six pollutants 0.717 (AUPRC 0.593) | forward fill 0.664 | adv +0.053 | Methods config 0.624
 50% | six pollutants 0.716 (AUPRC 0.592) | forward fill 0.645 | adv +0.071 [seed30 +0.062, +0.077] | Methods config 0.614
 60% | six pollutants 0.715 (AUPRC 0.591) | forward fill 0.618 | adv +0.097 | Methods config 0.603
 70% | six pollutants 0.715 (AUPRC 0.590) | forward fill 0.587 |

In [6]:
# calibration on the test months (all models fitted on the training months)
E_m6_tr,_,Ytr_e=score_bj(M6,rows="tr"); E_meth_tr,_,_=score_bj(METHODS,rows="tr")
assert (Ytr_e==ytr).all()
p_m6=platt_fit(E_m6_tr,ytr)(E_m6); p_meth=platt_fit(E_meth_tr,ytr)(E_meth); p_base=np.full(len(Y0),ytr.mean())
CAL=pd.DataFrame([
    ("estimator, six pollutants (recalibrated)", auc(p_m6,Y0),  apr(p_m6,Y0),  brier(p_m6,Y0),  ece(p_m6,Y0)),
    ("estimator, Methods config. (recalibrated)",auc(p_meth,Y0),apr(p_meth,Y0),brier(p_meth,Y0),ece(p_meth,Y0)),
    ("logistic, three signals",                  auc(p3,Y0),    apr(p3,Y0),    brier(p3,Y0),    ece(p3,Y0)),
    ("logistic, three signals + five pollutants",auc(p8,Y0),    apr(p8,Y0),    brier(p8,Y0),    ece(p8,Y0)),
    ("base rate only",                           np.nan,        np.nan,        brier(p_base,Y0),ece(p_base,Y0)),
],columns=["model","AUROC","AUPRC","Brier","ECE"]).set_index("model")
CAL.to_csv(OUT/"s25_beijing_calibration.csv"); print(CAL.round(4).to_string())

                                            AUROC   AUPRC   Brier     ECE
model                                                                    
estimator, six pollutants (recalibrated)   0.7242  0.5978  0.2315  0.0580
estimator, Methods config. (recalibrated)  0.6666  0.5235  0.2392  0.0393
logistic, three signals                    0.7360  0.6449  0.2042  0.0225
logistic, three signals + five pollutants  0.7543  0.6621  0.1978  0.0239
base rate only                                NaN     NaN  0.2444  0.0299


## Part B - Nordic network with NO2 as an additional input (threshold 15 ug/m3, six-hour means, 24 h horizon)
Data loading copied from study21. 19 of the 21 stations have NO2 at the same site; at the other two NO2 is missing
and is marginalised, as any missing observation.

In [7]:
TAU=15.0; STEP,H=6,4      # same aggregation and horizon as Part A, repeated so Part B can run on its own
pm=pd.read_csv("../data/openaq_regional/regional_pm25.csv"); pm["datetime"]=pd.to_datetime(pm["datetime"],utc=True)
pm["name"]=pm.station.str.split("::").str[1]; pm=pm.dropna(subset=["value"]); pm=pm[(pm.value>=0)&(pm.value<1500)]
met=pd.read_csv("../data/era5/era5_regional.csv",usecols=["datetime","station","wind_speed","blh","t2m"])
met["datetime"]=pd.to_datetime(met["datetime"],utc=True); met=met.rename(columns={"station":"name"})
yrs=pm.groupby("station")["datetime"].agg(lambda s:(s.max()-s.min()).days/365.25); pm=pm[pm.station.isin(yrs[yrs>=2].index)].copy()
no2=pd.read_csv("../data/openaq_regional/regional_no2.csv",usecols=["datetime","station","value"])
no2["datetime"]=pd.to_datetime(no2["datetime"],utc=True); no2=no2.dropna(subset=["value"]); no2=no2[(no2.value>=0)&(no2.value<1500)]
NO={}; n_with=0
for st,g in pm.groupby("station"):
    name=g["name"].iloc[0]
    s=g.set_index("datetime").value.resample(f"{STEP}h").mean()
    mg=met[met.name==name].set_index("datetime")[["wind_speed","blh","t2m"]].resample(f"{STEP}h").mean()
    idx=s.index; A=pd.DataFrame({"pm":s}).join(mg).reindex(idx); tm=(idx.month%2==0)
    n2=no2[no2.station==st]
    A["no2"]=n2.set_index("datetime").value.resample(f"{STEP}h").mean().reindex(idx) if len(n2) else np.nan
    n_with+=int(len(n2)>0)
    def z(col,log=False):
        v=A[col].to_numpy(dtype=float)
        if log: v=np.log1p(np.clip(v,0,None))
        if np.all(np.isnan(v[tm])): return np.full(len(v),np.nan)
        mu,sd=np.nanmean(v[tm]),np.nanstd(v[tm])+1e-9; return (v-mu)/sd
    v=A["pm"].to_numpy()
    Z=dict(zpm=z("pm",True),zw=z("wind_speed"),zb=z("blh"),zt=z("t2m"),zno2=z("no2",True))
    te=[];tr=[]
    for t in range(2,len(v)-H):
        if np.isnan(v[t]) or v[t]>=TAU: continue
        fut=v[t+1:t+1+H]
        if np.all(np.isnan(fut)): continue
        (tr if tm[t] else te).append((t,int(np.nanmax(fut)>=TAU)))
    NO[st]=dict(Z=Z,v=v,tm=tm,hr=idx.hour.to_numpy(),doy=idx.dayofyear.to_numpy(),te=te,tr=tr)
print(f"{len(NO)} Nordic stations, {n_with} with NO2 | test rows {sum(len(d['te']) for d in NO.values()):,}")
SPEC=[("zpm",1,1.0),("zw",-1,0.5),("zb",-1,0.5),("zt",-1,0.4)]
SPEC_NO2=SPEC+[("zno2",1,1.0)]
def score_no(cfg,frac=0.0,seed=None,rows="te"):
    rng=np.random.default_rng(seed) if frac>0 else None; E=[];Y=[]
    for st,d in NO.items():
        Z=dict(d["Z"]); zpm=Z["zpm"].copy()
        if frac>0: zpm[rng.random(len(zpm))<frac]=np.nan
        Z["zpm"]=zpm; risk=twin_cfg(Z,cfg)
        for t,y in d[rows]: E.append(risk[t]); Y.append(y)
    return np.array(E),np.array(Y,int)
E_spec,YN=score_no(SPEC); E_no2,_=score_no(SPEC_NO2)
print(f"check: specified configuration {auc(E_spec,YN):.3f} (Table 1: 0.794)")

/var/folders/hs/v3rv3f714391byj_w2dh_q980000gn/T/ipykernel_22669/1532861620.py:31: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  NO[st]=dict(Z=Z,v=v,tm=tm,hr=idx.hour.to_numpy(),doy=idx.dayofyear.to_numpy(),te=te,tr=tr)
/var/folders/hs/v3rv3f714391byj_w2dh_q980000gn/T/ipykernel_22669/1532861620.py:31: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  NO[st]=dict(Z=Z,v=v,tm=tm,hr=idx.hour.to_numpy(),doy=idx.dayofyear.to_numpy(),te=te,tr=tr)
/var/folders/hs/v3rv3f714391byj_w2dh_q980000gn/T/ipykernel_22669/1532861620.py:31: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  NO[st]=dict(Z=Z,v=v,tm=tm,hr=idx.hour.to_numpy(),doy=idx.dayofyear.to_numpy(),te=te,tr=tr)
/var/folders/hs/v3rv3f714391byj_w2dh_q980000gn/T/ipykernel_22669/15328

21 Nordic stations, 19 with NO2 | test rows 48,209
check: specified configuration 0.794 (Table 1: 0.794)


### Check 4 (Nordic) - specified configuration with and without NO2

In [8]:
d,lo,hi=paired_boot(E_no2,E_spec,YN); keep("4 Nordic NO2","specified + NO2 vs specified, complete",auc(E_no2,YN),auc(E_spec,YN),d,lo,hi)
print(f"complete data: specified {auc(E_spec,YN):.3f} (AUPRC {apr(E_spec,YN):.3f}) | + NO2 {auc(E_no2,YN):.3f} (AUPRC {apr(E_no2,YN):.3f}) | difference {fmt(d,lo,hi)}\n")
resN=[]
for fr in [0.4,0.7,0.85]:
    a=[];b=[]
    for s in [30,31,32]:
        Ea,Ya=score_no(SPEC,fr,s); Eb,_=score_no(SPEC_NO2,fr,s); a.append(auc(Ea,Ya)); b.append(auc(Eb,Ya))
        if s==30: dd,lo_,hi_=paired_boot(Eb,Ea,Ya)
    resN.append(dict(dropout=fr,specified=np.mean(a),with_no2=np.mean(b),diff=np.mean(b)-np.mean(a),diff_seed30=dd,lo95=lo_,hi95=hi_))
    keep("4 Nordic NO2",f"specified + NO2 vs specified, {fr*100:.0f}% PM2.5 masked",np.mean(b),np.mean(a),dd,lo_,hi_)
    print(f"{fr*100:3.0f}% PM2.5 masked | specified {np.mean(a):.3f} | + NO2 {np.mean(b):.3f} | diff {np.mean(b)-np.mean(a):+.3f} [seed30 {lo_:+.3f}, {hi_:+.3f}]")
pd.DataFrame(resN).to_csv(OUT/"s25_nordic_no2_dropout.csv",index=False)

complete data: specified 0.794 (AUPRC 0.254) | + NO2 0.782 (AUPRC 0.242) | difference -0.012 [-0.015, -0.009]

 40% PM2.5 masked | specified 0.767 | + NO2 0.760 | diff -0.007 [seed30 -0.010, -0.003]
 70% PM2.5 masked | specified 0.738 | + NO2 0.740 | diff +0.002 [seed30 -0.001, +0.007]
 85% PM2.5 masked | specified 0.721 | + NO2 0.729 | diff +0.008 [seed30 +0.004, +0.013]


### Check 1 (Nordic) - logistic model with and without NO2
Feature vector of the multimodal logistic baseline (study21), with the current NO2 value added (NaN set to 0).

In [9]:
g2=lambda a:(a if not np.isnan(a) else 0.0)
def no_X(rows,with_no2):
    X=[];Y=[]
    for d in NO.values():
        Z=d["Z"]; zc=Z["zpm"]; hr=d["hr"]; doy=d["doy"]
        for t,y in d[rows]:
            f=[g2(zc[t]),g2(zc[t-1]),g2(zc[t-2]),g2(zc[t]-zc[t-2]),g2(Z["zw"][t]),g2(Z["zb"][t]),g2(Z["zt"][t]),
               math.sin(2*math.pi*hr[t]/24),math.cos(2*math.pi*hr[t]/24),math.sin(2*math.pi*doy[t]/365),math.cos(2*math.pi*doy[t]/365)]
            if with_no2: f.append(g2(Z["zno2"][t]))
            X.append(f); Y.append(y)
    return np.array(X,float),np.array(Y,int)
Xa_tr,yN_tr=no_X("tr",False); Xa_te,yN_te=no_X("te",False); Xb_tr,_=no_X("tr",True); Xb_te,_=no_X("te",True)
qa=predlog(fitlog(Xa_tr,yN_tr),Xa_te); qb=predlog(fitlog(Xb_tr,yN_tr),Xb_te)
d,lo,hi=paired_boot(qb,qa,yN_te); keep("1 Nordic logistic","+ NO2 vs multimodal features",auc(qb,yN_te),auc(qa,yN_te),d,lo,hi)
print(f"logistic, multimodal features : AUROC {auc(qa,yN_te):.3f} (Table 1: 0.812)\nlogistic, + NO2               : AUROC {auc(qb,yN_te):.3f}   difference {fmt(d,lo,hi)}")

logistic, multimodal features : AUROC 0.812 (Table 1: 0.812)
logistic, + NO2               : AUROC 0.812   difference -0.000 [-0.000, +0.000]


In [10]:
R=pd.DataFrame(RESULTS); R.to_csv(OUT/"s25_summary.csv",index=False)
pd.set_option("display.width",200); print(R.round(4).to_string(index=False)); print("\nsaved to",OUT)

                   check                                         comparison  value_a  value_b    diff    lo95    hi95
      1 Beijing logistic                         +5 pollutants vs 3 signals   0.7543   0.7360  0.0183  0.0155  0.0211
       2 Beijing add-one                          Methods + PM10 vs Methods   0.6838   0.6666  0.0172  0.0130  0.0213
       2 Beijing add-one                           Methods + SO2 vs Methods   0.6948   0.6666  0.0282  0.0237  0.0326
       2 Beijing add-one                           Methods + NO2 vs Methods   0.6843   0.6666  0.0177  0.0139  0.0215
       2 Beijing add-one                            Methods + CO vs Methods   0.6775   0.6666  0.0109  0.0082  0.0136
       2 Beijing add-one                            Methods + O3 vs Methods   0.6436   0.6666 -0.0230 -0.0303 -0.0155
3 Beijing six pollutants    six (Methods weights) vs forward fill, complete   0.7242   0.7196  0.0045 -0.0010  0.0104
3 Beijing six pollutants         six (Methods weights) v

## Reading the results (send the outputs before any text is changed)

- **Check 3 is the deciding one.** If six pollutants with the Methods weights are above forward fill and close to
  the study24 variant, the claim can be stated as: the parameters are unchanged; where co-located pollutants are
  measured they enter as further observations of the same latent state. If they are not, the improvement depended on
  the feasibility-study weights and the claim must be narrowed.
- **Check 1** shows whether a model unrelated to the estimator also gains from the same inputs (information in the data).
- **Check 2** shows which pollutants carry the signal; combustion-related species (CO, NO2, PM10) would support a
  shared-source explanation, which still needs a verified reference before it is written.
- **Check 4** shows whether the effect is specific to Beijing. No gain in the Nordic network would support a
  regime-specific reading; a gain would make it a general finding.
- Caveats that apply whatever the outcome: these runs follow study24 and were specified after it; correlated
  pollutants are counted as independent evidence; the robustness benefit assumes the other analysers keep working.